# Importing Libraries


In [1]:
# Import Polars for dataframe operations
import polars as pl

# Import Path to manage fiel paths
from pathlib import Path

## Define Data Path


In [2]:
# Define path to the data folder
data_folder = Path("../data")

# Load Cleaned Dataset


In [3]:
# Load cleaned sales dataset
sales_df = pl.read_parquet(data_folder / "sales_cleaned.parquet")

# Preview Dataset
sales_df.head()

transaction_id,customer_id,product_id,purchase_date,festival,city,product_name,category,customer_name,age,gender,price,revenue
str,str,str,datetime[μs],str,str,str,str,str,i64,str,i64,i64
"""T82""","""c377""","""p3""",2024-01-01 00:00:00,null,"""Jorhat""","""Lip Balm""","""Makeup""","""Harshil Varty""",21,"""Female""",149,149
"""T910""","""c202""","""p1""",2024-01-01 00:00:00,null,"""Indore""","""Vitamin C Serum""","""Skincare""","""Tanvi Chana""",51,"""Male""",499,499
"""T1273""","""c752""","""p2""",2024-01-01 00:00:00,null,"""Secunderabad""","""Herbal Shampoo""","""Haircare""","""Idika Roy""",48,"""Male""",199,199
"""T1369""","""c739""","""p0""",2024-01-01 00:00:00,null,"""Bhatpara""","""Aloe Vera Gel""","""Skincare""","""Harita Comar""",49,"""Female""",299,299
"""T2858""","""c169""","""p1""",2024-01-01 00:00:00,null,"""Bhagalpur""","""Vitamin C Serum""","""Skincare""","""Zayyan Mander""",51,"""Female""",499,499


## Extract Purchase Month

skipped purchase_date to datetime since it already exists.


In [6]:
# Extract purchase month
sales_df = sales_df.with_columns(
    pl.col("purchase_date")
    .dt.truncate("1mo")
    .alias("purchase_month")
)

# Quick Check
sales_df.select(["purchase_date", "purchase_month"]).head()

purchase_date,purchase_month
datetime[μs],datetime[μs]
2024-01-01 00:00:00,2024-01-01 00:00:00
2024-01-01 00:00:00,2024-01-01 00:00:00
2024-01-01 00:00:00,2024-01-01 00:00:00
2024-01-01 00:00:00,2024-01-01 00:00:00
2024-01-01 00:00:00,2024-01-01 00:00:00


# Identify Customer First Purchase (Cohort)

Each customer belongs to the month they first purchased.


In [9]:
# Find first purchase month for each customer
cohort_df = sales_df.group_by("customer_id").agg(
    pl.col("purchase_month").min().alias("cohort_month")
)

cohort_df.head()

customer_id,cohort_month
str,datetime[μs]
"""c198""",2024-01-01 00:00:00
"""c183""",2024-01-01 00:00:00
"""c160""",2024-01-01 00:00:00
"""c749""",2024-01-01 00:00:00
"""c167""",2024-01-01 00:00:00


## Join Cohort Back to Transactions


In [ ]:
# Merge cohort info into sales data
sales_cohort_df = sales_df.join(cohort_df, on="customer_id")

# Quick Check
sales_cohort_df.head()

transaction_id,customer_id,product_id,purchase_date,festival,city,product_name,category,customer_name,age,gender,price,revenue,purchase_month,cohort_month
str,str,str,datetime[μs],str,str,str,str,str,i64,str,i64,i64,datetime[μs],datetime[μs]
"""T82""","""c377""","""p3""",2024-01-01 00:00:00,null,"""Jorhat""","""Lip Balm""","""Makeup""","""Harshil Varty""",21,"""Female""",149,149,2024-01-01 00:00:00,2024-01-01 00:00:00
"""T910""","""c202""","""p1""",2024-01-01 00:00:00,null,"""Indore""","""Vitamin C Serum""","""Skincare""","""Tanvi Chana""",51,"""Male""",499,499,2024-01-01 00:00:00,2024-01-01 00:00:00
"""T1273""","""c752""","""p2""",2024-01-01 00:00:00,null,"""Secunderabad""","""Herbal Shampoo""","""Haircare""","""Idika Roy""",48,"""Male""",199,199,2024-01-01 00:00:00,2024-01-01 00:00:00
"""T1369""","""c739""","""p0""",2024-01-01 00:00:00,null,"""Bhatpara""","""Aloe Vera Gel""","""Skincare""","""Harita Comar""",49,"""Female""",299,299,2024-01-01 00:00:00,2024-01-01 00:00:00
"""T2858""","""c169""","""p1""",2024-01-01 00:00:00,null,"""Bhagalpur""","""Vitamin C Serum""","""Skincare""","""Zayyan Mander""",51,"""Female""",499,499,2024-01-01 00:00:00,2024-01-01 00:00:00


## Calculating Cohort Index

cohort index = months since first purchase


In [ ]:
# Calculate months since first purchase
sales_cohort_df = sales_cohort_df.with_columns(
    (
        (pl.col("purchase_month").dt.year() -
         pl.col("cohort_month").dt.year()) * 12
        + (pl.col("purchase_month").dt.month() -
           pl.col("cohort_month").dt.month())
    ).alias("cohort_index")
)

sales_cohort_df.head()

transaction_id,customer_id,product_id,purchase_date,festival,city,product_name,category,customer_name,age,gender,price,revenue,purchase_month,cohort_month,cohort_index
str,str,str,datetime[μs],str,str,str,str,str,i64,str,i64,i64,datetime[μs],datetime[μs],i32
"""T82""","""c377""","""p3""",2024-01-01 00:00:00,null,"""Jorhat""","""Lip Balm""","""Makeup""","""Harshil Varty""",21,"""Female""",149,149,2024-01-01 00:00:00,2024-01-01 00:00:00,0
"""T910""","""c202""","""p1""",2024-01-01 00:00:00,null,"""Indore""","""Vitamin C Serum""","""Skincare""","""Tanvi Chana""",51,"""Male""",499,499,2024-01-01 00:00:00,2024-01-01 00:00:00,0
"""T1273""","""c752""","""p2""",2024-01-01 00:00:00,null,"""Secunderabad""","""Herbal Shampoo""","""Haircare""","""Idika Roy""",48,"""Male""",199,199,2024-01-01 00:00:00,2024-01-01 00:00:00,0
"""T1369""","""c739""","""p0""",2024-01-01 00:00:00,null,"""Bhatpara""","""Aloe Vera Gel""","""Skincare""","""Harita Comar""",49,"""Female""",299,299,2024-01-01 00:00:00,2024-01-01 00:00:00,0
"""T2858""","""c169""","""p1""",2024-01-01 00:00:00,null,"""Bhagalpur""","""Vitamin C Serum""","""Skincare""","""Zayyan Mander""",51,"""Female""",499,499,2024-01-01 00:00:00,2024-01-01 00:00:00,0


In [ ]:
# Quick Check
sales_cohort_df.select(
    ["purchase_month", "cohort_month", "cohort_index"]).tail(10)

purchase_month,cohort_month,cohort_index
datetime[μs],datetime[μs],i32
2024-12-01 00:00:00,2024-01-01 00:00:00,11
2024-12-01 00:00:00,2024-01-01 00:00:00,11
2024-12-01 00:00:00,2024-01-01 00:00:00,11
2024-12-01 00:00:00,2024-01-01 00:00:00,11
2024-12-01 00:00:00,2024-01-01 00:00:00,11
2024-12-01 00:00:00,2024-01-01 00:00:00,11
2024-12-01 00:00:00,2024-01-01 00:00:00,11
2024-12-01 00:00:00,2024-01-01 00:00:00,11
2024-12-01 00:00:00,2024-01-01 00:00:00,11


## Count Active Customers Per Cohort


In [16]:
# Count customers in each cohort period
cohort_counts = sales_cohort_df.group_by(
    ["cohort_month", "cohort_index"]
).agg(pl.col("customer_id").n_unique().alias("customers"))

cohort_counts.tail()

cohort_month,cohort_index,customers
datetime[μs],i32,u32
2024-03-01 00:00:00,8,7
2024-03-01 00:00:00,2,8
2024-01-01 00:00:00,4,911
2024-01-01 00:00:00,9,923
2024-01-01 00:00:00,8,873


## Get Cohort Sizes


In [ ]:
# Get initial cohort size
cohort_sizes = cohort_counts.filter(pl.col("cohort_index") == 0
                                    ).select(["cohort_month", "customers"]
                                             ).rename({"customers": "cohort_size"})

## Calculate Retention Rate


In [21]:
# Merge cohort size
cohort_retention = cohort_counts.join(
    cohort_sizes, on="cohort_month"
)

# Calculate retention rate
cohort_retention = cohort_retention.with_columns(
    (pl.col("customers") / pl.col("cohort_size")).alias("retention_rate")
)

cohort_retention.head()

cohort_month,cohort_index,customers,cohort_size,retention_rate
datetime[μs],i32,u32,u32,f64
2024-01-01 00:00:00,5,917,934,0.981799
2024-02-01 00:00:00,0,57,57,1.0
2024-06-01 00:00:00,4,1,1,1.0
2024-02-01 00:00:00,10,52,57,0.912281
2024-01-01 00:00:00,0,934,934,1.0


## Save Dataset


In [ ]:
# Save cohort retention dataset
cohort_retention.write_csv(data_folder / "cohort_retention.csv")

# Save parquet version
cohort_retention.write_parquet(data_folder / "cohort_retention.parquet")